In [2]:
print("Entrepreneur")

Entrepreneur


In [3]:
# Cell 1: Imports and project configuration

from pathlib import Path
import json
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Automatically locate the project root
current_path = Path.cwd()

possible_roots = [current_path, *current_path.parents]

PROJECT_ROOT = next(
    (
        path for path in possible_roots
        if (path / "data" / "application_train.csv").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate data/application_train.csv. "
        "Check that VS Code opened the notebook from your Credit_Risk project."
    )

DATA_PATH = PROJECT_ROOT / "data" / "application_train.csv"
OUTPUT_DIR = PROJECT_ROOT / "fine_tuning" / "data_splits"

RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)
print("Dataset exists:", DATA_PATH.exists())

Project root: c:\CODING\Credit_Risk
Dataset path: c:\CODING\Credit_Risk\data\application_train.csv
Output directory: c:\CODING\Credit_Risk\fine_tuning\data_splits
Dataset exists: True


In [4]:
# Cell 2: Load and inspect the dataset

# Check whether the file is a Git LFS pointer
with open(DATA_PATH, "rb") as file:
    file_header = file.read(200)

if file_header.startswith(b"version https://git-lfs.github.com/spec"):
    raise ValueError(
        "This is a Git LFS pointer, not the actual dataset. "
        "Run git lfs pull in your project directory."
    )

# Load dataset
df = pd.read_csv(DATA_PATH)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print("\nFirst 5 records:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.value_counts().rename_axis("Data Type").to_frame("Count"))

# Essential column checks
required_columns = {"TARGET", "SK_ID_CURR"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("\nRequired columns:")
print("TARGET:", "TARGET" in df.columns)
print("SK_ID_CURR:", "SK_ID_CURR" in df.columns)

# Target distribution
print("\nTARGET distribution:")
display(
    df["TARGET"]
    .value_counts(dropna=False)
    .rename("Count")
    .to_frame()
    .assign(
        Percentage=lambda x: (x["Count"] / len(df) * 100).round(2)
    )
)

# Data integrity
print("\nMissing TARGET values:", df["TARGET"].isna().sum())
print("Duplicate applicant IDs:", df["SK_ID_CURR"].duplicated().sum())
print("Duplicate complete rows:", df.duplicated().sum())

DATASET OVERVIEW
Rows: 307,511
Columns: 122
Memory usage: 325.22 MB

First 5 records:


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0



Column names:
['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMON

,Count
Data Type,
float64,65
int64,41
str,16



Required columns:
TARGET: True
SK_ID_CURR: True

TARGET distribution:


,Count,Percentage
TARGET,,
0,282686,91.93
1,24825,8.07



Missing TARGET values: 0
Duplicate applicant IDs: 0
Duplicate complete rows: 0


In [5]:
# Cell 3: Create stratified 70:15:15 split

# Stage 1: 70% training, 30% temporary
train_indices, temp_indices = train_test_split(
    df.index,
    test_size=0.30,
    stratify=df["TARGET"],
    random_state=RANDOM_STATE
)

# Stage 2: Split temporary data equally into validation and test
validation_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    stratify=df.loc[temp_indices, "TARGET"],
    random_state=RANDOM_STATE
)

# Extract partitions
train_df = df.loc[train_indices].copy()
validation_df = df.loc[validation_indices].copy()
test_df = df.loc[test_indices].copy()

# Display partition sizes
print("PARTITION SIZES")
print("-" * 50)

for name, partition in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    print(
        f"{name:12} | "
        f"Rows: {len(partition):,} | "
        f"Percentage: {len(partition) / len(df) * 100:.2f}%"
    )

# Compare target distributions
print("\nTARGET DISTRIBUTION BY PARTITION")
print("-" * 50)

distribution = pd.DataFrame({
    "Full Dataset": df["TARGET"].value_counts(normalize=True) * 100,
    "Training": train_df["TARGET"].value_counts(normalize=True) * 100,
    "Validation": validation_df["TARGET"].value_counts(normalize=True) * 100,
    "Test": test_df["TARGET"].value_counts(normalize=True) * 100
}).sort_index()

distribution.index = ["Non-default (0)", "Default (1)"]

display(distribution.round(2))

PARTITION SIZES
--------------------------------------------------
Training     | Rows: 215,257 | Percentage: 70.00%
Validation   | Rows: 46,127 | Percentage: 15.00%
Test         | Rows: 46,127 | Percentage: 15.00%

TARGET DISTRIBUTION BY PARTITION
--------------------------------------------------


,Full Dataset,Training,Validation,Test
Non-default (0),91.93,91.93,91.93,91.93
Default (1),8.07,8.07,8.07,8.07


In [6]:
# Cell 4: Verify partition integrity

train_ids = set(train_df["SK_ID_CURR"])
validation_ids = set(validation_df["SK_ID_CURR"])
test_ids = set(test_df["SK_ID_CURR"])

# Check pairwise overlaps
overlap_checks = {
    "Train vs Validation": train_ids.intersection(validation_ids),
    "Train vs Test": train_ids.intersection(test_ids),
    "Validation vs Test": validation_ids.intersection(test_ids)
}

print("APPLICANT OVERLAP CHECK")
print("-" * 50)

for comparison, overlapping_ids in overlap_checks.items():
    print(f"{comparison}: {len(overlapping_ids)} overlapping applicants")

# Check whether all original applicants are accounted for
combined_ids = train_ids | validation_ids | test_ids

print("\nDATA COMPLETENESS CHECK")
print("-" * 50)

print("Original unique applicants:", df["SK_ID_CURR"].nunique())
print("Applicants across partitions:", len(combined_ids))
print("Missing applicants:", len(set(df["SK_ID_CURR"]) - combined_ids))

# Automated assertions
assert all(len(ids) == 0 for ids in overlap_checks.values()), \
    "Applicant overlap detected!"

assert len(combined_ids) == df["SK_ID_CURR"].nunique(), \
    "Some applicants are missing!"

assert len(train_df) + len(validation_df) + len(test_df) == len(df), \
    "Partition row counts do not match original dataset!"

print("\nAll partition integrity checks passed.")

APPLICANT OVERLAP CHECK
--------------------------------------------------
Train vs Validation: 0 overlapping applicants
Train vs Test: 0 overlapping applicants
Validation vs Test: 0 overlapping applicants

DATA COMPLETENESS CHECK
--------------------------------------------------
Original unique applicants: 307511
Applicants across partitions: 307511
Missing applicants: 0

All partition integrity checks passed.


In [7]:
# Cell 5: Save V2 partitions and split metadata

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save partitions
train_path = OUTPUT_DIR / "train.csv"
validation_path = OUTPUT_DIR / "validation.csv"
test_path = OUTPUT_DIR / "test.csv"

train_df.to_csv(train_path, index=False)
validation_df.to_csv(validation_path, index=False)
test_df.to_csv(test_path, index=False)

# Prepare reproducibility metadata
metadata = {
    "experiment": "Credit Risk V2",
    "dataset": "Home Credit application_train.csv",
    "random_state": RANDOM_STATE,
    "split_strategy": "Stratified two-stage train_test_split",
    "split_proportions": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15
    },
    "partitions": {
        "train": {
            "rows": len(train_df),
            "default_rate": round(float(train_df["TARGET"].mean()), 6)
        },
        "validation": {
            "rows": len(validation_df),
            "default_rate": round(float(validation_df["TARGET"].mean()), 6)
        },
        "test": {
            "rows": len(test_df),
            "default_rate": round(float(test_df["TARGET"].mean()), 6)
        }
    },
    "total_rows": len(df),
    "total_columns": len(df.columns),
    "identifier_column": "SK_ID_CURR",
    "target_column": "TARGET",
    "model_input_exclusions": ["SK_ID_CURR", "TARGET"]
}

metadata_path = OUTPUT_DIR / "split_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

# Confirm saved files
print("V2 DATA SPLITS SAVED")
print("-" * 50)

for path in [train_path, validation_path, test_path, metadata_path]:
    print(f"{path.name}: {path.stat().st_size / (1024**2):.2f} MB")

print("\nSaved location:", OUTPUT_DIR)

V2 DATA SPLITS SAVED
--------------------------------------------------
train.csv: 110.48 MB
validation.csv: 23.66 MB
test.csv: 23.64 MB
split_metadata.json: 0.00 MB

Saved location: c:\CODING\Credit_Risk\fine_tuning\data_splits
